In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu langchain langchain-text-splitters langchain_community
!pip install -q fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
from pyngrok import ngrok
!ngrok config add-authtoken 3AjLmLIfO9VLukducyThBaeh50h_6k9LFSKxeT8w4e9RgvC7U


public_url = ngrok.connect(8000)

print("Your public API URL:")
print(public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Your public API URL:
NgrokTunnel: "https://loyce-unlamentable-nonpleadingly.ngrok-free.dev" -> "http://localhost:8000"


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict
from pyngrok import ngrok

# QA

In [4]:
#7B MODEL FOR QA

qa_model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_id)

qa_model = AutoModelForCausalLM.from_pretrained(
    qa_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [5]:
embeddings_model= HuggingFaceEmbeddings(
      model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
     )

/tmp/ipykernel_1284/3202515024.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model= HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def build_vector_store_from_text(text: str):
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    docs = splitter.create_documents([text])
    # embeddings = HuggingFaceEmbeddings(
    #     model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # )
    # 3. Create the Payload
    payload = []
    for i, doc in enumerate(docs):
        # Generate the vector as a list of floats
        vector = embeddings_model.embed_query(doc.page_content)
        payload.append({
            "content": doc.page_content,
            "embedding": vector,
            "chunkIndex": i
        })

    # Return the raw data instead of the FAISS object
    return payload
    #return FAISS.from_documents(docs, embeddings)


In [7]:
def build_messages_with_rag(message_history: List[Dict], retrieved_context: str):
    """
    Builds the final prompt using the context provided by the Java backend.
    """
    # 1. Identify the latest user message to wrap it in the RAG prompt
    latest_user_index = next(
        i for i in reversed(range(len(message_history)))
        if message_history[i]["role"] == "user"
    )
    latest_user_message = message_history[latest_user_index]["content"]

    # 2. Keep the history leading up to the current question
    previous_messages = message_history[:latest_user_index]

    # 3. Create the specialized RAG prompt
    # Note: We use 'retrieved_context' which was passed in as a string
    rag_user_message = {
        "role": "user",
        "content": f"""
Answer the question using ONLY the provided document content.

Rules:
- Base your answer strictly on the document content.
- Do not invent information.
- If the information does not exist at all, say:
  Sorry, this information was not mentioned in the documents.
- Answer in the same language as the question.
- Keep the answer concise.
- If you were asked about a previous response from you, you can answer.

Document Content:
{retrieved_context}

Question:
{latest_user_message}
"""
    }

    # 4. Combine previous chat history with the new context-heavy message
    return previous_messages + [rag_user_message]

In [8]:
def generate_response(message_history: List[Dict], max_new_tokens=256):
    messages = [{"role": "system", "content": "You are a strict question answering system."}] + message_history
    text = qa_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qa_tokenizer(text, return_tensors="pt").to(qa_model.device)

    outputs = qa_model.generate(**inputs, max_new_tokens=max_new_tokens, repetition_penalty=1.1)
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return qa_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

**Initiate the Fast API**

In [9]:
app = FastAPI(title="RAG QA API")



**Fetch and Rebuild The Vectorestore form the DB**

**Initiate Chat Endpoint**

In [10]:
class InitChatRequest(BaseModel):
    transcript: str # Renamed to match your Java 'InitRequest' DTO

@app.post("/chat/{chat_id}/init")
def init_chat(chat_id: str, request: InitChatRequest):
    try:
        # 1. Use the function we already have to split text and create vectors
        # This returns a List of: {"content": str, "embedding": List[float], "chunkIndex": int}

        payload = build_vector_store_from_text(request.transcript)



        response = {
            "reply": "Embeddings generated successfully.",
            "payload": payload
        }

        return response
    except Exception as e:
        print(f"Error during init: {e}")
        import traceback
        traceback.print_exc()
        return {"error": str(e)}


In [11]:
@app.post("/embed")
def embed_text(request: dict):
    # Returns the list of 384 floats
    return embeddings_model.embed_query(request.get("text"))

In [12]:
from pydantic import BaseModel
from typing import List

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    message: str
    history: List[Message] = []
    context: List[str] = [] # These are the chunks Java found in pgvector
@app.post("/chat/{chat_id}/ask")
def chat_ask(chat_id: str, request: ChatRequest):
    # 1. Generate the "Payload" (Embeddings) for the NEW message
    # Java needs this to save the current message into pgvector
    payload = build_vector_store_from_text(request.message)

    # 2. Convert the Pydantic history objects into simple dictionaries for the AI
    history_dicts = [{"role": m.role, "content": m.content} for m in request.history]

    # 3. Add the current message to the working history
    history_dicts.append({"role": "user", "content": request.message})

    # 4. Join the context strings Java sent from the local DB
    # We pass this string directly into the RAG message builder
    retrieved_context = "\n\n".join(request.context)

    # 5. Build messages and Generate Response
    # Note: build_messages_with_rag now takes a STRING instead of a retriever
    final_messages = build_messages_with_rag(history_dicts, retrieved_context)
    reply = generate_response(final_messages)

    # 6. Return everything to Java
    return {
        "reply": reply,
        "payload": payload # Java will loop through this to save to PGVector
    }

**Ask Endpoint**

In [13]:
!pip install nest_asyncio

In [14]:
import uvicorn
import asyncio

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Since Colab has a loop running, we just schedule the server task
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

<Task pending name='Task-4' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>